## Prediction for the full run (from reading the scripts, before running)
- Startup imports should now succeed (pydub, mutagen and astral are installed here; `config` is found because run_pipeline.py sits next to config.py).
- Steps 1 to 5 should run in order and succeed: 1,747 clips, then Tiny CNN, BirdNET, acoustic features, metadata.
- Step 6 should crash: `get_weather_info_all()` reads `AUDIO_MOTH_FOLDER` from config.py, which is still "path/to/raw", so I expect FileNotFoundError, before the 'Datetime' KeyError we saw earlier.
- Steps 7 and 8 would not run.

In [1]:
from nb_paths import PIPELINE_DIR, KWF_TEST_DIR
import subprocess, sys
from pathlib import Path

HOME = Path.home()
RUN_DIR = KWF_TEST_DIR / "pipeline_run"

result = subprocess.run(
    [sys.executable, "run_pipeline.py",
     "--raw_audio", str(KWF_TEST_DIR / "raw_audio"),
     "--output", str(RUN_DIR)],
    cwd=PIPELINE_DIR, capture_output=True, text=True,
)
print("Exit code:", result.returncode)
print("\n--- last 40 lines of output ---")
print("\n".join((result.stdout + result.stderr).splitlines()[-40:]))

print("\n--- files produced ---")
for p in sorted((RUN_DIR / "outputs").glob("*")):
    print(p.name)
print("clips written:", len(list((RUN_DIR / "clips").glob("*.wav"))))

Exit code: 1

--- last 40 lines of output ---
Inference: 100%|██████████| 1747/1747 [00:25<00:00, 67.25it/s]
2026-10-21 16:02:59,830 — INFO — Acoustic features saved to: /Users/huyguy/kwf_test/pipeline_run/outputs/acoustic_features.csv
2026-10-21 16:02:59,831 — INFO — STEP 5 — Metadata extraction...
2026-10-21 16:03:12,564 — INFO — Metadata saved to: /Users/huyguy/kwf_test/pipeline_run/outputs/metadata.csv
2026-10-21 16:03:12,564 — INFO — STEP 6 — Weather data fetch + matching...
Traceback (most recent call last):
  File "/Users/huyguy/Documents/Documents - Huys MacBook Pro/Job Materials/KwF/Inference Pipeline/KWF_Human_Presence/inference_pipeline/run_pipeline.py", line 217, in <module>
    main()
  File "/Users/huyguy/Documents/Documents - Huys MacBook Pro/Job Materials/KwF/Inference Pipeline/KWF_Human_Presence/inference_pipeline/run_pipeline.py", line 207, in main
    weather_csv          = step6_weather(outputs_dir)
                           ^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/User

What this does: runs run_pipeline.py exactly as the team’s usage line says (--raw_audio and --output), in a separate process so a crash can’t take down the notebook, and prints everything it logs. The output goes to a new folder, ~/kwf_test/pipeline_run, and the input is the raw-audio folder from the Step 1 test.

How to read the output: the log shows each step starting (“STEP 1 — ...”). The last step line before a traceback is where the pipeline stopped. The traceback’s last line names the error. Then the cell lists which output files exist, which shows how far it got.

Steps 1 to 5 ran through run_pipeline.py without stopping. Step 4 finished all 1,747 clips (25 s), and Step 5 wrote metadata.csv in about 13 s.
Step 6 is where it breaks, as predicted. Step 6 ignored the --raw_audio folder we passed and went to the placeholder path in config.py.
Step 7 and Step 8 never ran.

In [2]:
lines = (result.stdout + result.stderr).splitlines()
print("Final line:", lines[-1] if lines else "(no output)")

print("\n--- step markers and summaries ---")
for l in lines:
    if any(k in l for k in ["STEP", "Meaningful clips", "Clips saved", "Done.", "Skipped", "Processed", "Metadata saved"]):
        print(l)

print("\n--- files produced ---")
for p in sorted((RUN_DIR / "outputs").glob("*")):
    print(p.name)
print("clips written:", len(list((RUN_DIR / "clips").glob("*.wav"))))

Final line: FileNotFoundError: [Errno 2] No such file or directory: 'path/to/raw'

--- step markers and summaries ---
Processed 20250319_163237.WAV: 1747 clips
Skipped 0 missing/corrupt clips.
Done. 1,747 clips processed, 101 with detections.
Done. 1,747 rows saved, 0 failed.
Metadata saved to /Users/huyguy/kwf_test/pipeline_run/outputs/metadata.csv
2026-10-21 16:01:39,749 — INFO — STEP 1 — Splitting raw audio into 3-second clips...
2026-10-21 16:01:44,179 — INFO — Clips saved to: /Users/huyguy/kwf_test/pipeline_run/clips
2026-10-21 16:01:44,179 — INFO — STEP 2 — Tiny CNN: filtering meaningful clips...
2026-10-21 16:01:56,176 — INFO — Meaningful clips: 1,747 / 1,747 total
2026-10-21 16:01:56,176 — INFO — STEP 3 — Compressed BirdNET: species + confidence...
2026-10-21 16:02:32,371 — INFO — STEP 4 — Acoustic feature extraction...
2026-10-21 16:02:59,831 — INFO — STEP 5 — Metadata extraction...
2026-10-21 16:03:12,564 — INFO — Metadata saved to: /Users/huyguy/kwf_test/pipeline_run/outputs

## Observations: run 1 (run_pipeline.py as shipped)

**Command:** `python run_pipeline.py --raw_audio ~/kwf_test/raw_audio --output ~/kwf_test/pipeline_run` (1 raw file, 87.4 min, recorder 5).

**Result:** exit code 1. Steps 1 to 5 ran: 1,747 clips; Tiny CNN 1,747/1,747 meaningful; BirdNET 101 clips with detections; acoustic features 1,747 rows, 0 failed; metadata written. Counts match the individual step tests.

**Stopped at Step 6:** `FileNotFoundError: 'path/to/raw'`. Step 6 reads its folders from `config.py` (still placeholders) and ignores `--raw_audio`. Steps 7 and 8 did not run; no `weather.csv` or `features.csv` was produced.

## Prediction for run 2 (patched scratch copy)
- Exit code 0. Steps 1 to 5 as before.
- Step 6 finds the raw file and downloads 2 weather hours (16:00, 17:00); 599 clips get blank weather.
- Step 7 writes features.csv with 1,747 rows (all meaningful), 34 columns, and the same values as my notebook-07 run (nb07_features.csv).
- Step 8 only logs "HP Model not yet available. Skipping inference." (placeholder).

In [3]:
from nb_paths import PIPELINE_DIR, KWF_TEST_DIR
import shutil, subprocess, sys, tempfile
from pathlib import Path

HOME = Path.home()
RUN2 = KWF_TEST_DIR / "pipeline_run2"
shutil.rmtree(RUN2, ignore_errors=True)
(RUN2 / "outputs").mkdir(parents=True)

scratch = Path(tempfile.mkdtemp(prefix="pipeline_scratch_")) / "inference_pipeline"
shutil.copytree(PIPELINE_DIR, scratch, ignore=shutil.ignore_patterns("__pycache__", "pipeline.log", "testing"))

def patch(path, old, new):
    text = path.read_text()
    assert text.count(old) == 1, f"{path.name}: expected 1 match for {old!r}, found {text.count(old)}"
    path.write_text(text.replace(old, new))
    print("PATCHED OK:", path.name, "|", old[:55])

cfg = scratch / "config.py"
patch(cfg, 'AUDIO_MOTH_FOLDER = r"path/to/raw"', f'AUDIO_MOTH_FOLDER = r"{KWF_TEST_DIR / "raw_audio"}"')
patch(cfg, 'r"path/to/weather_data.csv"', f'r"{RUN2 / "outputs" / "weather_data.csv"}"')
patch(cfg, 'r"path/to/weather_matches.csv"', f'r"{RUN2 / "outputs" / "weather.csv"}"')
patch(cfg, 'r"path/to/metadata.csv"', f'r"{RUN2 / "outputs" / "metadata.csv"}"')
patch(scratch / "scripts" / "weather_matching.py",
      "df_weather = pd.read_csv(config.WEATHER_DATA_PATH)",
      'df_weather = pd.read_csv(config.WEATHER_DATA_PATH).rename(columns={"datetime": "Datetime", "temperature": "Temperature", "windspeed": "Windspeed", "humidity": "Humidity"})')

result2 = subprocess.run(
    [sys.executable, "run_pipeline.py",
     "--raw_audio", str(KWF_TEST_DIR / "raw_audio"),
     "--output", str(RUN2)],
    cwd=scratch, capture_output=True, text=True,
)
lines = (result2.stdout + result2.stderr).splitlines()
print("\nExit code:", result2.returncode)
print("Final line:", lines[-1] if lines else "(no output)")
print("\n--- step markers ---")
for l in lines:
    if any(k in l for k in ["STEP", "Meaningful clips", "Kept ", "Done.", "WARNING", "Error", "ERROR", "Combined weather"]):
        print(l)
print("\n--- files produced ---")
for p in sorted((RUN2 / "outputs").glob("*")):
    print(p.name)

shutil.rmtree(scratch.parent)

PATCHED OK: config.py | AUDIO_MOTH_FOLDER = r"path/to/raw"
PATCHED OK: config.py | r"path/to/weather_data.csv"
PATCHED OK: config.py | r"path/to/weather_matches.csv"
PATCHED OK: config.py | r"path/to/metadata.csv"
PATCHED OK: weather_matching.py | df_weather = pd.read_csv(config.WEATHER_DATA_PATH)

Exit code: 0
Final line: 2026-10-21 20:28:32,219 — INFO — ============================================================

--- step markers ---
Done. 1,747 clips processed, 101 with detections.
Done. 1,747 rows saved, 0 failed.
[OK] Combined weather data saved to /Users/huyguy/kwf_test/pipeline_run2/outputs/weather_data.csv
Kept 1,747 AED-meaningful clips.
Done. 1,747 rows saved to /Users/huyguy/kwf_test/pipeline_run2/outputs/features.csv
2026-10-21 20:26:42,947 — INFO — STEP 1 — Splitting raw audio into 3-second clips...
2026-10-21 20:26:48,529 — INFO — STEP 2 — Tiny CNN: filtering meaningful clips...
2026-10-21 20:27:00,257 — INFO — Meaningful clips: 1,747 / 1,747 total
2026-10-21 20:27:00,25

What this does: copies inference_pipeline to a temporary folder, applies five small text patches to the copy (four config paths and one column rename), checks each patch applied exactly once, and runs run_pipeline.py from the copy into a new output folder.

How to read the output: “PATCHED OK” lines mean each replacement matched. Exit code 0 means the whole pipeline ran. The step markers show how far it got, and the file list should now include weather.csv and features.csv. Any new error is the next blocker in the chain, and its last line is the finding.

## Prediction for run 3 (main at 4104a17, no patches)
- Exit code 0. Steps 1 to 5 give the same counts as before (code unchanged): 1,747 clips, 1,747/1,747 meaningful, BirdNET 101 clips with detections, acoustic 1,747 rows, metadata 1,747 rows.
- Step 6 logs the two new files (weather_hourly.csv, weather.csv). The hourly file should have more than the 2 hours we saw before (probably 3: 16:00, 17:00, 18:00), because of the "extra hour" fix.
- weather.csv: 1,747 rows with 0 blank weather values (was 599 blank).
- features.csv: 1,747 rows, 34 columns, 0 blanks in the model features. Non-weather columns identical to nb07_features.csv.
- Step 8 only logs the placeholder warning.

In [4]:
from nb_paths import PIPELINE_DIR, KWF_TEST_DIR
import subprocess, sys, shutil
from pathlib import Path

HOME = Path.home()
RUN3 = KWF_TEST_DIR / "pipeline_run3"
shutil.rmtree(RUN3, ignore_errors=True)

git = lambda *a: subprocess.run(["git", *a], cwd=PIPELINE_DIR, capture_output=True, text=True).stdout.strip()
print("Code version:", git("rev-parse", "--short", "HEAD"), "| branch:", git("branch", "--show-current"))

result3 = subprocess.run(
    [sys.executable, "run_pipeline.py",
     "--raw_audio", str(KWF_TEST_DIR / "raw_audio"),
     "--output", str(RUN3)],
    cwd=PIPELINE_DIR, capture_output=True, text=True,
)
lines = (result3.stdout + result3.stderr).splitlines()
print("\nExit code:", result3.returncode)
print("Final line:", lines[-1] if lines else "(no output)")

print("\n--- step markers ---")
for l in lines:
    if any(k in l for k in ["STEP", "Meaningful clips", "Kept ", "Done.", "Hourly weather", "Per-clip weather",
                            "WARNING", "ERROR", "Error", "Weather   :"]):
        print(l)

print("\n--- files produced ---")
for p in sorted((RUN3 / "outputs").glob("*")):
    print(p.name)
print("clips written:", len(list((RUN3 / "clips").glob("*.wav"))))

Code version: 4104a17 | branch: main

Exit code: 0
Final line: 2026-10-21 20:41:42,607 — INFO — ============================================================

--- step markers ---
Done. 1,747 clips processed, 101 with detections.
Done. 1,747 rows saved, 0 failed.
Kept 1,747 AED-meaningful clips.
Done. 1,747 rows saved to /Users/huyguy/kwf_test/pipeline_run3/outputs/features.csv
2026-10-21 20:39:48,748 — INFO — Weather   : on
2026-10-21 20:39:48,749 — INFO — STEP 1 — Splitting raw audio into 3-second clips...
2026-10-21 20:39:54,725 — INFO — STEP 2 — Tiny CNN: filtering meaningful clips...
2026-10-21 20:40:06,808 — INFO — Meaningful clips: 1,747 / 1,747 total
2026-10-21 20:40:06,809 — INFO — STEP 3 — Compressed BirdNET: species + confidence...
2026-10-21 20:40:51,774 — INFO — STEP 4 — Acoustic feature extraction...
2026-10-21 20:41:23,880 — INFO — STEP 5 — Metadata extraction...
2026-10-21 20:41:41,656 — INFO — STEP 6 — Weather data fetch + matching...
2026-10-21 20:41:42,505 — INFO — Ho

What this does: runs run_pipeline.py from the repo on main with the same arguments as run 1 and no patches, into a new output folder (pipeline_run3). It first prints the git commit, so the notebook records which version of the code was tested. A subprocess starts a fresh Python process, so it loads the updated files with no leftovers from earlier cells.

How to read the output: the commit should be 4104a17. Exit code 0 means the pipeline finished. The step markers show how far it got, and the “Hourly weather” and “Per-clip weather” lines show Step 6 writing its new files. The file list include weather_hourly.csv, weather.csv and features.csv. Any error’s last line names the next blocker.

In [5]:
from nb_paths import PIPELINE_DIR, KWF_TEST_DIR
import pandas as pd
O1 = KWF_TEST_DIR / "outputs"
O3 = KWF_TEST_DIR / "pipeline_run3" / "outputs"

hourly = pd.read_csv(O3 / "weather_hourly.csv")
print("Hourly weather:", hourly.shape)
show = [c for c in ["datetime", "Datetime", "temperature", "Temperature", "precipitation", "humidity", "Humidity", "weather_desc"] if c in hourly.columns]
print(hourly[show].to_string(index=False))

w = pd.read_csv(O3 / "weather.csv")
print("\nPer-clip weather:", w.shape, "| unique clips:", w["clip_name"].nunique())
print("Columns:", w.columns.tolist())
print("Blank values per column (non-zero only):", w.isna().sum()[w.isna().sum() > 0].to_dict() or "none")

def compare(label, f1, f2, drop=()):
    a = pd.read_csv(O1 / f1).drop(columns=list(drop)).sort_values("clip_name").reset_index(drop=True)
    b = pd.read_csv(O3 / f2).drop(columns=list(drop)).sort_values("clip_name").reset_index(drop=True)
    same_cols = a.columns.tolist() == b.columns.tolist()
    same_clips = same_cols and a["clip_name"].equals(b["clip_name"])
    print(f"\n{label}: shape {a.shape} vs {b.shape} | same columns {same_cols} | same clips {same_clips}")
    if same_clips:
        num = a.select_dtypes("number").columns
        txt = [c for c in a.columns if c not in num]
        print("   max numeric diff:", float((a[num] - b[num]).abs().max().max()),
              "| blanks same:", a[num].isna().equals(b[num].isna()),
              "| text same:", all(a[c].equals(b[c]) for c in txt))

compare("Step 2 Tiny CNN", "nb02_recorder5_rerun_predictions.csv", "tinycnn_predictions.csv", drop=["audio_path"])
compare("Step 3 BirdNET", "nb03_birdnet_output.csv", "birdnet_output.csv")
compare("Step 4 acoustic", "nb04_recorder5_acoustic_features.csv", "acoustic_features.csv")
compare("Step 5 metadata", "nb05_metadata.csv", "metadata.csv")

Hourly weather: (3, 11)
           Datetime  Temperature  precipitation  Humidity weather_desc
2025-03-19 16:00:00         27.8            0.0        76    Clear sky
2025-03-19 17:00:00         27.1            0.0        80    Clear sky
2025-03-19 18:00:00         26.1            0.0        86    Clear sky

Per-clip weather: (1747, 12) | unique clips: 1747
Columns: ['Datetime', 'Temperature', 'Windspeed', 'precipitation', 'Humidity', 'weathercode', 'weather_desc', 'timestamp_local', 'sunrise', 'sunset', 'time_of_day', 'clip_name']
Blank values per column (non-zero only): none

Step 2 Tiny CNN: shape (1747, 5) vs (1747, 5) | same columns True | same clips True
   max numeric diff: 0.0 | blanks same: True | text same: True

Step 3 BirdNET: shape (1747, 3) vs (1747, 3) | same columns True | same clips True
   max numeric diff: nan | blanks same: True | text same: True

Step 4 acoustic: shape (1747, 11) vs (1747, 11) | same columns True | same clips True
   max numeric diff: 0.0 | blanks s

What this does: inspects the two new weather files (rows, columns, blanks, which hours they cover), then compares run 3’s Step 2 to 5 outputs with the notebook versions we validated earlier.

How to read the output: 3 hourly rows (16:00, 17:00 and 18:00, thanks to the “extra hour” fix) and 0 blank weather values (it was 599). Steps 2 to 5 are unchanged code, so each comparison show “same clips True”, “max diff 0.0”, “blanks same True” and “text same True”. The Step 2 audio_path column is dropped, because its folder differs between runs.

In [6]:
f3 = pd.read_csv(O3 / "features.csv").sort_values("clip_name").reset_index(drop=True)
f1 = pd.read_csv(O1 / "nb07_features.csv").sort_values("clip_name").reset_index(drop=True)
wcols = ["Temperature", "Windspeed", "Humidity", "Volume_Wind_Ratio"]

print("Run 3 features:", f3.shape, "| earlier hand run:", f1.shape)
print("Same columns:", f1.columns.tolist() == f3.columns.tolist(), "| same clips:", f1["clip_name"].equals(f3["clip_name"]))
if f1.columns.tolist() == f3.columns.tolist() and f1["clip_name"].equals(f3["clip_name"]):
    other = [c for c in f1.columns if c not in wcols]
    num = f1[other].select_dtypes("number").columns
    txt = [c for c in other if c not in num]
    print("Non-weather numeric max diff:", float((f1[num] - f3[num]).abs().max().max()),
          "| text same:", all(f1[c].equals(f3[c]) for c in txt))
print("Blank weather values:", f3[wcols].isna().sum().to_dict())

f3["Datetime"] = pd.to_datetime(f3["Datetime"])
print("\nWhich clip times got which temperature:")
print(f3.groupby("Temperature")["Datetime"].agg(["min", "max", "count"]).to_string())

Run 3 features: (1747, 34) | earlier hand run: (1747, 34)
Same columns: True | same clips: True
Non-weather numeric max diff: 0.0 | text same: True
Blank weather values: {'Temperature': 0, 'Windspeed': 0, 'Humidity': 0, 'Volume_Wind_Ratio': 0}

Which clip times got which temperature:
                            min                 max  count
Temperature                                               
26.1        2025-03-19 17:30:01 2025-03-19 17:59:55    599
27.1        2025-03-19 16:32:37 2025-03-19 17:29:58   1148


What this does: compares run 3’s features.csv with our earlier hand-run nb07_features.csv (the one made with a renamed weather copy), and shows which clip times got which temperature.

How to read the output: the non-weather columns should be identical. Blank weather values í 0 (it was 599). The last table shows the matching rule: each temperature value with the first and last clip time that received it.

## Observations: run 3 (main at 4104a17, run_pipeline.py with no patches)

**Command:** same as run 1. **Result:** exit code 0; all 8 steps ran in about 114 s on 1 raw file (87.4 min, recorder 5). Step 8 only logs the placeholder warning.

**Compared with the hand-run notebooks:** Steps 2 to 5 outputs identical (same clips, max numeric difference 0.0, same blanks and text). Step 7 non-weather columns identical to the earlier hand run.

**Step 6 (new code):** hourly weather has 3 rows (16:00, 17:00, 18:00); per-clip weather has 1,747 rows and 0 blanks (run 1/2 code: 599 blanks). features.csv: 1,747 rows, 34 columns, 0 blanks in Temperature, Windspeed, Humidity, Volume_Wind_Ratio.

**Fixed since run 1:** Step 6 paths and --raw_audio, weather.csv written, weather column names, blank weather at the end of a recording.

**Still open:** matching is nearest-hour (clips 16:32 to 17:29 get the 17:00 weather; the 16:00 row is never used) and hour_sin uses the clip's own hour; weather.csv column casing is mixed (Temperature vs precipitation).

**Not tested:** --no_weather, lowercase ".wav" raw files, several recorders in one hour, the HP model.

## Prediction for run 4 (--no_weather, main at 4104a17)
- Exit code 0; Step 6 logged as skipped; no weather_hourly.csv or weather.csv.
- features.csv: 1,747 rows, 30 columns = run 3's 34 minus Temperature, Windspeed, Humidity, Volume_Wind_Ratio (25 model features + 5 extras).
- All remaining columns identical to run 3.

In [7]:
from nb_paths import PIPELINE_DIR, KWF_TEST_DIR
RUN4 = KWF_TEST_DIR / "pipeline_run4"
shutil.rmtree(RUN4, ignore_errors=True)
result4 = subprocess.run(
    [sys.executable, "run_pipeline.py", "--raw_audio", str(KWF_TEST_DIR / "raw_audio"),
     "--output", str(RUN4), "--no_weather"],
    cwd=PIPELINE_DIR, capture_output=True, text=True,
)
lines = (result4.stdout + result4.stderr).splitlines()
print("Exit code:", result4.returncode, "| final line:", lines[-1] if lines else "")
print([l for l in lines if "STEP 6" in l or "Weather   :" in l])
print("Files:", sorted(p.name for p in (RUN4 / "outputs").glob("*")))

f4 = pd.read_csv(RUN4 / "outputs" / "features.csv").sort_values("clip_name").reset_index(drop=True)
f3 = pd.read_csv(O3 / "features.csv").sort_values("clip_name").reset_index(drop=True)
print("\nShape:", f4.shape, "| columns missing vs run 3:", [c for c in f3.columns if c not in f4.columns])
same = f3[f4.columns]
num = f4.select_dtypes("number").columns
print("Remaining columns identical to run 3:",
      float((same[num] - f4[num]).abs().max().max()) == 0.0 and all(same[c].equals(f4[c]) for c in f4.columns if c not in num))

Exit code: 0 | final line: 2026-10-21 20:49:41,907 — INFO — ============================================================
['2026-10-21 20:47:54,064 — INFO — Weather   : off (--no_weather)', '2026-10-21 20:49:41,819 — INFO — STEP 6 — Skipped (--no_weather)']
Files: ['acoustic_features.csv', 'birdnet_output.csv', 'features.csv', 'metadata.csv', 'tinycnn_predictions.csv']

Shape: (1747, 30) | columns missing vs run 3: ['Temperature', 'Windspeed', 'Humidity', 'Volume_Wind_Ratio']
Remaining columns identical to run 3: True


What this does: runs the same command as run 3 with --no_weather added, into a new folder, and compares the resulting features.csv with run 3’s.

How to read the output: “Step 6 skipped” in the markers and no weather files confirm the flag works. The list of columns that disappeared should be exactly the four weather-based ones. “Remaining columns identical” means the flag didn’t change anything else. It reruns Steps 1 to 5, so it takes about 2 minutes.

## Observations: run 4 (--no_weather, main at 4104a17)

**Result:** exit code 0 (about 108 s). Log: "Weather: off (--no_weather)" and "STEP 6 — Skipped"; no weather_hourly.csv or weather.csv written. features.csv: 1,747 rows, 30 columns. Exactly four columns are absent versus run 3 (Temperature, Windspeed, Humidity, Volume_Wind_Ratio); all remaining columns are identical to run 3.

**Not tested:** that the 25 model features match the team's training feature list.

**Code version:** the findings in this notebook were obtained against commit `4f86604`, before the weather fix (PR #2). Run 3 in `99_end_to_end_run_pipeline.ipynb` (commit `4104a17`) shows which of them are fixed: the weather column names, the Step 6 paths, and the blank weather values.